# RetinaGuard — Train & Evaluate on APTOS 2019 (Kaggle)

**Setup before running (right-hand sidebar → Settings / Session options):**
1. **Accelerator → GPU** (T4 x2 or P100).
2. **Internet → ON** (needed for `git clone` + `pip`).
3. **Add Input** (top-right '+ Add Input' → Competitions tab → search `aptos2019-blindness-detection` → Add). You must have joined the competition once: https://www.kaggle.com/c/aptos2019-blindness-detection/rules

Then run every cell top to bottom (each cell separately — do not merge).

## 1. Confirm GPU + locate the APTOS data

In [ ]:
import torch, os, glob
print('CUDA available:', torch.cuda.is_available())
# Kaggle mounts added competition data under /kaggle/input/
APTOS = '/kaggle/input/aptos2019-blindness-detection'
print('APTOS contents:', os.listdir(APTOS))
IMAGES_DIR = os.path.join(APTOS, 'train_images')
CSV_PATH   = os.path.join(APTOS, 'train.csv')
print('images:', len(glob.glob(IMAGES_DIR + '/*')), '| csv exists:', os.path.exists(CSV_PATH))

## 2. Get the RetinaGuard code (clone the repo)
Requires Internet = ON. Pulls the latest pipeline incl. the fixed train.py / evaluate.py.

In [ ]:
%cd /kaggle/working
![ -d RetinaGuard ] && rm -rf RetinaGuard
!git clone https://github.com/Gaurav-0704/RetinaGuard.git
%cd RetinaGuard
# sanity: confirm evaluate.py is complete (last line must call evaluate(...))
!tail -n 1 ml/evaluate.py
!python -c "import ast; ast.parse(open('ml/train.py').read()); ast.parse(open('ml/evaluate.py').read()); print('train.py + evaluate.py parse OK')"

## 3. Install any missing deps
Kaggle already ships torch/sklearn/pandas/matplotlib. This just ensures opencv is present.

In [ ]:
!pip -q install opencv-python-headless 2>/dev/null
import cv2, sklearn, matplotlib; print('cv2', cv2.__version__, '| sklearn', sklearn.__version__)

## 4. (Recommended first) Quick smoke test — 2 epochs
Confirms the whole pipeline runs end-to-end on the GPU before you commit to the full run. It temporarily sets NUM_EPOCHS=2 via an env-free edit, trains, and evaluates. If this produces a confusion matrix, the full run will work.

In [ ]:
# Temporarily shrink epochs for the smoke test by patching config in memory is not enough
# (train runs as a subprocess), so edit the config file, run, then restore.
!sed -i.bak 's/^NUM_EPOCHS *=.*/NUM_EPOCHS = 2/' ml/config.py
!python -m ml.train --images_dir $IMAGES_DIR --csv_path $CSV_PATH
!python -m ml.evaluate --images_dir $IMAGES_DIR --csv_path backend/weights/test_split.csv
!mv ml/config.py.bak ml/config.py   # restore full epoch count
print('Smoke test done. If you see a classification report above, the pipeline works.')

## 5. Full training run
Best checkpoint (by validation QWK) saved to backend/weights/retinoguard_best.pth; held-out test split to backend/weights/test_split.csv. On a T4/P100 expect roughly 30-60 min.

In [ ]:
!python -m ml.train --images_dir $IMAGES_DIR --csv_path $CSV_PATH

## 6. Evaluate on the held-out test split

In [ ]:
!python -m ml.evaluate --images_dir $IMAGES_DIR --csv_path backend/weights/test_split.csv

## 7. Show the results inline

In [ ]:
print(open('docs/results/report.md').read())
from IPython.display import Image, display
for p in ['docs/results/confusion_matrix.png','docs/results/confusion_matrix_normalized.png','docs/results/roc_curves.png']:
    display(Image(p))

## 8. Save outputs for download
Copies weights + results into /kaggle/working (the Output tab). After the session, open the notebook's **Output** tab to download `retinoguard_best.pth`, `metrics.json`, `report.md`, and the PNGs.

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/outputs', exist_ok=True)
for src in ['backend/weights/retinoguard_best.pth','backend/weights/history.json','backend/weights/test_split.csv']:
    if os.path.exists(src): shutil.copy(src, '/kaggle/working/outputs/')
if os.path.isdir('docs/results'):
    shutil.copytree('docs/results', '/kaggle/working/outputs/results', dirs_exist_ok=True)
print('Saved to /kaggle/working/outputs:', os.listdir('/kaggle/working/outputs'))

## 9. Use the results locally
Download the weights + results from the Output tab. Put `retinoguard_best.pth` at `backend/weights/` in your repo, paste the numbers from `report.md` into `docs/RESULTS.md`, and update the README.

**Sanity check:** strong single-model APTOS results land around QWK 0.88-0.91 / accuracy ~0.82-0.85. QWK > 0.95 suggests a data leak — investigate before reporting.